In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import *
import matplotlib.pyplot as plt
import nltk

In [ ]:
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df.head()

In [ ]:
df.info()

# How to tweets look like in both the classes ?
- using basic cleaning techniques like lowercasing the texts and removing the punctuation
- then visualizing how text in each class looks

## BaseLine Modelling
- Linear Models perform better than Tree based models on sparse datasets - Biggest Learning from here

In [ ]:
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
stop_words = set(stopwords.words('english'))

In [ ]:
df['target'].value_counts(normalize=True)

In [ ]:
def clean_words(text):
    #lower casing all the text
    lower = text.lower()
    #removing urls
    text = re.sub(r'https?://\S+|www\.\S+','',lower)
    #remove punctuations
    text = re.sub(r'[^a-z\s]','',text)
    #finally the stopwords
    text = " ".join(word for word in text.split() if word not in stop_words)
    return text
output = clean_words("Can't believe this man!! just set ablaze")
print(f'Cleaned_text: {output}')

In [ ]:
df['Cleaned_Text'] = df['text'].apply(clean_words)
print(f'Shape of Cleaned Text: {df.shape}')

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000,ngram_range=(1,2))
X = vectorizer.fit_transform(df['Cleaned_Text'])
y = df['target']

print(f'Shape of feature matrix: {X.shape}')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import *
import sklearn
model = LogisticRegression()

scores = cross_val_score(model,X,y,cv=5,scoring='f1',n_jobs=-1)
print(f'Scores: {scores}\nMean: {np.mean(scores)}')

## Going beyond Words and Using better models
- instead of using simply the words, we'll construct more features from the dataset

### Proposed Features (yet)
1. tweet lengths
2. word counts
3. punctuation counts
4. Exclamation mark counts
5. hashtag counts
6. '@' count
#### more features
7. unique word count
8. stop word count
9. UpperCase word count
10. Sentence count
11. average word lengths
12. avg sentence length
13. Digit counts

### On the Model
- using better models like **SupportVectorMachines** which have been shown to work the best with **SparseData**

In [ ]:
import string
def PreProcess(data,target='text'):
    df = data.copy()
    n_feat = f'{target}_cleaned'
    df['Length'] = df[target].apply(len)
    df['Words'] = df[target].apply(lambda text: len(text.split()))
    df['PCount'] = df[target].apply(lambda text: sum(1 for char in text if char in string.punctuation))
    df['#_count'] = df[target].apply(lambda text: text.count("#"))
    df['@_count'] = df[target].apply(lambda text: text.count("@"))
    df['!_count'] = df[target].apply(lambda text: text.count("!"))
    ##New Features
    df['UWords'] = df[target].apply(lambda text: len(set(text.split())))
    df['SWords'] = df[target].apply(lambda text: len([words for words in text.split() if words in stop_words]))
    df['UpperWords'] = df[target].apply(lambda text: len([words for words in text.split() if words.isupper()]))
    # df['Sentences'] = df[target].apply(lambda text: len(text.split("\n")))
    df['avg_word_length'] = df['Length'] / df['Words']
    
    df[n_feat] =df[target].apply(clean_words)
    return df

In [ ]:
data = PreProcess(df)
data.head(3)

In [ ]:
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler
vectorizer = TfidfVectorizer(max_features=10000,ngram_range=(1,2))

X_tfidf = vectorizer.fit_transform(data['text_cleaned'])
scaler = StandardScaler()
meta_features = data[['Length','Words','PCount','#_count','@_count','!_count','UWords','SWords','UpperWords','avg_word_length']]
meta_features = scaler.fit_transform(meta_features)
X = hstack([X_tfidf,meta_features])
y = df.target

In [ ]:
print(f'Training Shape: {X.shape}')

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
model1 = LogisticRegression(
    class_weight='balanced',
    C=0.3,
    max_iter=2500,
    random_state=9100
)

model2 = SVC(
    kernel='linear',
    C=0.5,
    class_weight='balanced',
    cache_size=5*2048,
    random_state=42
)

model3 = GradientBoostingClassifier(subsample=0.8,random_state=42)

In [ ]:
score1 = cross_val_score(model1,X,y,cv=5,scoring='f1',n_jobs=-1)
print(f'Scores:{score1}, Mean:{np.mean(score1)}')
score2 = cross_val_score(model2,X,y,cv=5,scoring='f1',n_jobs=-1)
print(f'Scores:{score2}, Mean:{np.mean(score2)}')
score3 = cross_val_score(model3,X,y,cv=5,scoring='f1',n_jobs=-1)
print(f'Scores:{score3}, Mean:{np.mean(score3)}')

In [ ]:
model1.fit(X,y)

# Submission

In [ ]:
testing = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
testing = PreProcess(testing)
testing.head(3)

In [ ]:
test_idf = vectorizer.transform(testing['text_cleaned'])
# scaler = StandardScaler()
meta_features = testing[['Length','Words','PCount','#_count','@_count','!_count','UWords','SWords','UpperWords','avg_word_length']]
meta_features = scaler.transform(meta_features)
X = hstack([test_idf,meta_features])
# y = df.target

In [ ]:
preds = model1.predict(X)

In [ ]:
pd.DataFrame({
    'id':testing.id,
    'target':preds
}).to_csv('submission.csv',index=False)